# 🗓️ 27일차 스터디 노트북 — KMP법 (기억하는 검색)

**오늘 범위**: 07-2 KMP법 — 겹치는 문자열 찾기 · 건너뛰기 표(skip table) 만들기 · 실습 7-2 `kmp_match` · KMP의 한계

## 난이도 태그
🟢 **기본** (전원 필수) / 🟡 **표준** (팀 목표선) / 🔴 **심화** (도전)

## 유형 태그
**[손]** · **[빈칸]** · **[예측]** · **[구현]** · **[디버깅]** · **[설명]** · **[실험]** · **[추적]**

---

## 어제 남긴 숙제

26일차 7번에서 이걸 증명했지:

```python
pt = pt - pp + 1     # 새 시작 위치 = 기존 시작 위치 + 1
```

그리고 10번에서 `pt = pt + 1` 로 바꾸면 **여러 칸을 근거 없이 건너뛰어 정답을 놓친다**는 걸 봤어.

> **오늘의 질문: 그럼 "근거 있게" 여러 칸을 건너뛰려면 어떻게 해야 할까?** 🔥

교재 310p가 답을 줘:
> **"브루트 포스법은 일치하지 않는 문자를 만나면 이전 단계에서 검사했던 결과를 버리고 패턴의 첫 문자부터 다시 검사를 수행합니다. 하지만 KMP법은 검사했던 결과를 버리지 않고 효율적으로 활용하는 알고리즘입니다."**

**"검사했던 결과를 버리지 않는다"** — 이게 오늘 전부야.

## 브루트 포스 vs KMP 한눈에

| | 불일치했을 때 |
|---|---|
| 브루트 포스 (26일) | `pt`를 **되돌리고** `pp = 0` |
| **KMP (오늘)** | `pt`는 **그대로**, `pp`만 **줄인다** |

`pt`가 되돌아가지 않는다 = **텍스트를 두 번 읽지 않는다.** 교재 314p가 이걸 **"브루트 포스법에는 없는 특징"** 이라고 콕 집어 말해.

## 진행 순서
**개념(1~3) → 건너뛰기 표 손으로 만들기(4~7) → 검색 손 추적(8~10) → 코드 구현(11~15) → 성질과 한계(16~18)**

표를 **손으로 만드는** 문제가 많아. 종이랑 펜 준비.

> 📁 아래 **"부록"** 셀을 **가장 먼저 실행**해.

---

## 📁 부록 — 오늘의 도구 (제일 먼저 실행!)

In [ ]:
import time, random

# 교재 실습 7-2
def kmp_match(txt: str, pat: str) -> int:
    """KMP법으로 문자열 검색"""
    pt = 1                        # txt를 따라가는 커서
    pp = 0                        # pat를 따라가는 커서
    skip = [0] * (len(pat) + 1)   # 건너뛰기 표

    # 건너뛰기 표 만들기
    skip[pt] = 0
    while pt != len(pat):
        if pat[pt] == pat[pp]:
            pt += 1; pp += 1; skip[pt] = pp
        elif pp == 0:
            pt += 1; skip[pt] = pp
        else:
            pp = skip[pp]

    # 문자열 검색하기
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]:
            pt += 1; pp += 1
        elif pp == 0:
            pt += 1
        else:
            pp = skip[pp]

    return pt - pp if pp == len(pat) else -1


def build_skip(pat: str) -> list:
    """건너뛰기 표만 따로 만들기 (실험용)"""
    pt = 1; pp = 0
    skip = [0] * (len(pat) + 1)
    skip[pt] = 0
    while pt != len(pat):
        if pat[pt] == pat[pp]: pt += 1; pp += 1; skip[pt] = pp
        elif pp == 0: pt += 1; skip[pt] = pp
        else: pp = skip[pp]
    return skip


def max_overlap(s):
    """s의 앞부분과 뒷부분이 같은 최대 길이 (전체 제외)"""
    for L in range(len(s)-1, 0, -1):
        if s[:L] == s[len(s)-L:]:
            return L, s[:L]
    return 0, ""


def show_skip(pat):
    """건너뛰기 표를 교재 형식으로 출력"""
    sk = build_skip(pat)
    print("        " + "  ".join(pat))
    print("  skip  " + "  ".join(str(sk[i]) for i in range(1, len(pat)+1)))
    print(f"  (교재 표기로는 첫 칸이 '-' — 15번에서 설명)")
    return sk

# 26일차 브루트 포스 (비교용)
def bf_match(txt, pat):
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return pt - pp if pp == len(pat) else -1

def naive(t, p):
    for i in range(len(t) - len(p) + 1):
        if t[i:i+len(p)] == p: return i
    return -1

def align(txt, pat, start):
    """텍스트 위에 패턴을 start 위치에 놓고 그린다"""
    print(f"  idx  {' '.join(str(i%10) for i in range(len(txt)))}")
    print(f"  txt  {' '.join(txt)}")
    print(f"  pat  {'  '*start}{' '.join(pat)}")

TXT = 'ZABCABXACCADEF'
PAT = 'ABCABD'

print("준비 완료 ✅\n")
align(TXT, PAT, 0)
print()
show_skip(PAT)

---
# 🔁 [Remind] 워밍업 — 26일차 되감기

오늘은 어제 코드를 **한 줄만 바꾸는** 이야기야. 그 한 줄을 정확히 기억해야 해.

### R-1. 🟢 [설명] 어제의 `pt - pp` 불변식

```python
# 26일차 bf_match
else:
    pt = pt - pp + 1
    pp = 0
```

- 26일차 7번에서 증명한 것: `pt - pp` 는 항상 ①________________ 다.
- 불일치 시 이 두 줄이 하는 일을 한 문장으로: ②________________
- 그리고 26일차 10번에서 `pt = pt + 1` 로 바꾸면 시작 위치가 **`pp + 1` 칸** 튀어서 정답을 놓쳤지. 랜덤 3000회 중 몇 %가 틀렸어?

### R-2. 🟡 [예측] 오늘 바뀌는 곳

오늘 코드는 이렇게 생겼어:
```python
elif pp == 0:
    pt += 1
else:
    pp = skip[pp]     # ← pt는 안 건드린다!
```

- 어제와 비교해서 **`pt`에 무슨 일이 생겼지?**
- `pt`가 절대 줄어들지 않는다면, `while` 루프는 최대 몇 번 돌 수 있을까? (힌트: `pt`가 매번 커지거나 `pp`가 줄어들거나)
- 그럼 시간 복잡도는 O(n·m)에서 **O(?)** 로 바뀔까? 예측만 해두고 16번에서 확인해.

*(여기에 답 작성)*

---
# 🎯 PART 1 — 왜 되돌아가지 않아도 되나 (1~3번)

> 교재 310p의 그림 설명을 따라간다. **코드는 아직 안 봐.**

### 1. 🟢 [손] 교재 310p 시나리오

```
  idx  0 1 2 3 4 5 6 7 8 9 ...
  txt  Z A B C A B X A C C A D E F
  pat  A B C A B D
```

**1단계**: 첫 글자 `Z` vs `A` → 불일치. 패턴을 1칸 민다.

**2단계**: 이제 `txt[1]`부터 대조.
```
  txt  Z A B C A B X A C C ...
  pat    A B C A B D
```
- 몇 글자가 일치해? ①____
- 어디서 불일치가 나? `txt[____]` = ②____ vs `pat[____]` = ③____

**핵심 관찰** 🔥 — 교재 310p:
> **"여기서 파란색 문자로 나타낸 텍스트 안의 'AB'와 패턴 안의 'AB'가 일치하는 것에 주목합니다."**

- 지금까지 맞춘 부분은 `ABCAB` 야. 이 문자열의 **앞 2글자**와 **뒤 2글자**를 각각 적어봐.
  - 앞 2글자: ④____
  - 뒤 2글자: ⑤____
- 두 개가 같지? 그럼 **패턴을 3칸 밀면** 패턴의 `AB`가 텍스트의 (이미 맞춰본) `AB` 자리에 정확히 겹쳐.
- 그래서 교재는 **"패턴을 단 한 번에 오른쪽으로 3칸 밀어 3번째 문자 'C'부터 검사"** 한다고 해.
- 💡 **`AB`를 다시 비교할 필요가 없어.** 이미 맞다는 걸 알고 있으니까. 이게 "검사했던 결과를 버리지 않는다"의 뜻이야.

*(답을 적은 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
print("2단계: 패턴을 1칸 민 상태")
align(txt, pat, 1)
print()
matched = 0
while matched < len(pat) and 1 + matched < len(txt) and txt[1+matched] == pat[matched]:
    matched += 1
print(f"  일치한 글자 수: {matched}  → 맞춘 부분 = '{pat[:matched]}'")
print(f"  불일치: txt[{1+matched}]='{txt[1+matched]}' vs pat[{matched}]='{pat[matched]}'")

s = pat[:matched]
print(f"\n맞춘 부분 '{s}' 의 앞뒤 겹침 찾기:")
for L in range(len(s)-1, 0, -1):
    a, b = s[:L], s[len(s)-L:]
    mark = "  ← 가장 긴 겹침! ✅" if a == b else ""
    print(f"  앞 {L}글자 '{a}' vs 뒤 {L}글자 '{b}'  {'일치' if a==b else '불일치'}{mark}")
    if a == b: break

print(f"\n→ 패턴을 {matched - L}칸 밀면 '{a}'가 그대로 겹친다")
print(f"→ 즉 {L}글자는 비교를 건너뛰고, pat[{L}]부터 다시 검사하면 된다")
align(txt, pat, 1 + (matched - L))

### 2. 🟡 [설명] "겹치는 문자열"의 정체

1번에서 찾은 것: `ABCAB` 의 **앞 2글자 `AB`** 와 **뒤 2글자 `AB`** 가 같다.

- 이걸 일반화하면: **맞춘 부분의 "앞부분"과 "뒷부분"이 같은 최대 길이**를 찾는 거야.
- ⚠️ 단, **전체 길이는 안 돼.** `ABCAB` 의 앞 5글자와 뒤 5글자는 당연히 같지만, 그건 아무것도 안 미는 거니까 무의미하지.
- 몇 개 연습해봐. 각 문자열에서 **"앞부분 == 뒷부분"인 최대 길이**(전체 제외)를 구해:

| 문자열 | 최대 겹침 길이 | 그 문자열 |
|---|---|---|
| `A` | ① | |
| `AB` | ② | |
| `ABC` | ③ | |
| `ABCA` | ④ | |
| `ABCAB` | ⑤ | |
| `AAAA` | ⑥ | |
| `ABAB` | ⑦ | |
| `AABAA` | ⑧ | |

- 이 값이 곧 **"불일치했을 때 pp를 어디로 되돌릴지"** 야.
- 💡 이 값을 매번 계산하면 느리겠지? 교재 311p: **"몇 번째 문자부터 검사를 다시 시작할지 패턴을 이동할 때마다 계산한다면 좋은 효율을 기대할 수 없습니다. 그래서 KMP법은 '몇 번째 문자부터 다시 검색할지' 값을 표로 만들어서 문제를 해결합니다."**
  → 그 표가 **건너뛰기 표(skip table)** 야.

*(답을 적은 뒤 실행)*

In [ ]:
for s in ['A', 'AB', 'ABC', 'ABCA', 'ABCAB', 'AAAA', 'ABAB', 'AABAA']:
    L, sub = max_overlap(s)
    print(f"  '{s:6s}' → 최대 겹침 {L}  {'(' + sub + ')' if sub else ''}")

print("\n💡 이 값이 곧 skip 표의 값이 된다 (4번에서 확인)")

### 3. 🟢 [설명] 교재 그림 7-4 읽기

교재 311p [그림 7-4]는 패턴 `ABCABD`가 **몇 번째 문자에서 실패했느냐**에 따라 어디부터 다시 시작할지를 보여줘.

| 실패한 위치 | 그때까지 맞춘 부분 | 다시 시작할 위치 |
|---|---|---|
| ⓐ 1번째 문자 | (없음) | 1번째 문자부터 |
| ⓑ 2번째 문자 | `A` | 1번째 문자부터 |
| ⓒ 3번째 문자 | `AB` | 1번째 문자부터 |
| ⓓ 4번째 문자 | `ABC` | ① |
| ⓔ 5번째 문자 | `ABCA` | ② **2번째 문자부터** |
| ⓕ 6번째 문자 | `ABCAB` | ③ **3번째 문자부터** |

- ①을 채워봐. 그리고 ⓑ~ⓓ가 전부 "1번째 문자부터"인 이유를 2번의 표로 설명해봐.
- ⓔ에서 맞춘 부분은 `ABCA`야. 앞뒤 겹침이 `A`(길이 1)이니 **1글자는 건너뛰고 2번째 문자부터** 검사. 맞지?
- ⓕ에서 맞춘 부분은 `ABCAB`. 앞뒤 겹침이 `AB`(길이 2)이니 **2글자를 건너뛰고 3번째 문자부터**.
- 💡 **"다시 시작할 위치(1-based)" = "겹침 길이 + 1"** 이야. 0-based 인덱스로는 그냥 **겹침 길이**지. 이게 `pp = skip[pp]` 의 의미야.

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
print(f"pat = {pat}\n")
print("실패위치 | 맞춘 부분 | 겹침 | 다시 시작(0-based) | 다시 시작(1-based)")
print("-" * 70)
for k in range(1, len(pat)+1):
    s = pat[:k]
    L, _ = max_overlap(s)
    print(f"   {k}번째  | {s:9s} |  {L}   |      pat[{L}]         |    {L+1}번째")

print("\n→ '다시 시작(0-based)' 열이 곧 skip 표다!")

---
# 📋 PART 2 — 건너뛰기 표 손으로 만들기 (4~7번)

> 교재 312~313p. **패턴 두 개를 위아래로 놓고 1칸씩 밀면서** 만든다.
> 오늘 노트북에서 손을 가장 많이 쓰는 파트야.

### 4. 🟢 [손] 교재 방식 — 패턴끼리 겹쳐보기

교재 312p: **"패턴 'ABCABD' 2개를 위아래로 나란히 놓고 아래쪽 패턴을 오른쪽으로 1칸 밀어 서로 겹칩니다."**

```
A B C A B D
  A B C A B D      ← 1칸 밀기
```

**1. 1칸 밀었을 때** — 겹치는 부분은 `BCABD` vs `ABCAB` 의 앞부분
- 위 `B` vs 아래 `A` → 일치? ①____
- 첫 글자부터 안 맞으니 **2번째 문자 `B`의 값은 0**

```
A B C A B D
    A B C A B D    ← 2칸 밀기
```
**2. 2칸 밀었을 때**
- 위 `C` vs 아래 `A` → ②____ → **3번째 문자 `C`의 값은 ③____**

```
A B C A B D
      A B C A B D  ← 3칸 밀기
```
**3. 3칸 밀었을 때** 🔥
- 위 `A B` vs 아래 `A B` → ④____ ! **2글자가 겹친다**
- 교재 313p: **"패턴의 4번째 문자 'A'까지 일치하는 경우: 패턴 이동 후 'A'를 건너뛰고(skip) 2번째 문자부터 검사할 수 있습니다"** → 4번째 문자 `A`의 값은 ⑤____
- **"패턴의 5번째 문자 'B'까지 일치하는 경우: 'AB'를 건너뛰고 3번째 문자부터"** → 5번째 문자 `B`의 값은 ⑥____

```
A B C A B D
        A B C A B D  ← 4칸 밀기
```
**4. 4칸 밀었을 때**
- 위 `B` vs 아래 `A` → 불일치 → 끝 문자 `D`의 값은 ⑦____

**완성된 표를 채워봐:**
```
       A   B   C   A   B   D
skip   -   __  __  __  __  __
```

*(채운 뒤 실행해서 대조)*

In [ ]:
pat = PAT
print(f"패턴: {pat}\n")
for shift in range(1, len(pat)):
    top = pat
    bot = " " * shift + pat
    over_len = len(pat) - shift
    a, b = pat[shift:], pat[:over_len]
    ok = a == b
    print(f"{shift}칸 밀기:")
    print(f"  {' '.join(top)}")
    print(f"  {'  '*shift}{' '.join(pat)}")
    print(f"  겹치는 부분: 위 '{a}' vs 아래 '{b}' → {'✅ 일치' if ok else '❌ 불일치'}")
    if ok:
        print(f"  → 패턴의 {len(pat)}번째까지 맞았을 때 {over_len}글자 건너뛸 수 있음")
    print()

print("=" * 40)
sk = show_skip(pat)
print(f"\n실제 skip 리스트: {sk}  (크기 = len(pat)+1 = {len(sk)})")

### 5. 🟡 [손] 다른 패턴으로 직접 만들어보기

이제 혼자 만들어봐. **2번의 "앞뒤 겹침" 방법**을 쓰면 빨라.

각 패턴에 대해 `skip[1] ~ skip[m]` 을 채워:

**(가) `AAAA`**

| k (맞춘 길이) | 맞춘 부분 | 앞뒤 겹침 | skip[k] |
|---|---|---|---|
| 1 | `A` | 0 | 0 |
| 2 | `AA` | ① | ② |
| 3 | `AAA` | ③ | ④ |
| 4 | `AAAA` | ⑤ | ⑥ |

**(나) `ABAB`**

| k | 맞춘 부분 | 겹침 | skip[k] |
|---|---|---|---|
| 1 | `A` | | |
| 2 | `AB` | | |
| 3 | `ABA` | | |
| 4 | `ABAB` | | |

**(다) `ABCDE`** — 겹치는 게 하나도 없는 패턴

| k | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| skip[k] | | | | | |

**(라) `AABAAC`** 🔴

| k | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| skip[k] | | | | | | |

- (다)처럼 skip이 **전부 0**이면 KMP는 어떻게 동작할까? 브루트 포스와 뭐가 달라?
- (가)처럼 skip이 **계단식으로 증가**하면? 이건 어떤 텍스트에서 유리할까?

*(채운 뒤 실행)*

In [ ]:
for pat_ in ['AAAA', 'ABAB', 'ABCDE', 'AABAAC']:
    print(f"패턴 '{pat_}'")
    sk = build_skip(pat_)
    print(f"   k  맞춘부분    겹침  skip[k]")
    for k in range(1, len(pat_)+1):
        s = pat_[:k]
        L, _ = max_overlap(s)
        flag = "✅" if L == sk[k] else "❌"
        print(f"   {k}  {s:10s}  {L}     {sk[k]}   {flag}")
    print()

### 6. 🔴 [설명] 표 만들기 코드가 왜 KMP를 닮았나

```python
pt = 1
pp = 0
skip[pt] = 0
while pt != len(pat):
    if pat[pt] == pat[pp]:      # ← 패턴 vs 패턴!
        pt += 1; pp += 1; skip[pt] = pp
    elif pp == 0:
        pt += 1; skip[pt] = pp
    else:
        pp = skip[pp]           # ← 만들고 있는 표를 자기가 참조!
```

이 코드를 아래쪽 검색 코드와 나란히 놓고 봐:

```python
while pt != len(txt) and pp != len(pat):
    if txt[pt] == pat[pp]:      # ← 텍스트 vs 패턴
        pt += 1; pp += 1
    elif pp == 0:
        pt += 1
    else:
        pp = skip[pp]
```

- **구조가 거의 똑같지?** 뭐가 다르고 뭐가 같아?
- 교재 312p: **"표를 작성할 때는 패턴에서 겹치는 문자열을 찾습니다. 이 과정에서도 KMP법과 같은 방법을 적용합니다."**
  → **"패턴 안에서 패턴을 검색하는 것"** 이라고 볼 수 있어. 왜 그렇게 볼 수 있는지 설명해봐.
- 🔥 `pp = skip[pp]` 에서 **아직 만들고 있는 중인 표를 참조**해. 이게 가능한 이유는?
  💡 힌트: `skip[pp]` 를 읽을 때 `pp < pt` 인데, `skip[1] ~ skip[pt]` 는 이미 채워졌잖아?
- `pt = 1`, `pp = 0` 으로 시작하는 이유는? 왜 둘 다 0이 아닐까?

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
print(f"표 만들기 단계별 추적 (pat = {pat})\n")
pt = 1; pp = 0
skip = [0] * (len(pat) + 1)
skip[pt] = 0
print(f"  초기: pt={pt}, pp={pp}, skip={skip}")
step = 0
while pt != len(pat):
    step += 1
    if pat[pt] == pat[pp]:
        branch = f"① 일치 pat[{pt}]='{pat[pt]}' == pat[{pp}]='{pat[pp]}'"
        pt += 1; pp += 1; skip[pt] = pp
        detail = f"pt,pp 둘 다 +1, skip[{pt}]={pp}"
    elif pp == 0:
        branch = f"② 불일치 & pp==0: pat[{pt}]='{pat[pt]}' != pat[0]='{pat[0]}'"
        pt += 1; skip[pt] = pp
        detail = f"pt만 +1, skip[{pt}]=0"
    else:
        branch = f"③ 불일치 & pp!=0: pp를 skip[{pp}]={skip[pp]} 로 되돌림"
        pp = skip[pp]
        detail = "pt는 그대로! (읽기 위치를 안 되돌린다)"
    print(f"  {step}: {branch}")
    print(f"     → {detail}")
    print(f"     pt={pt}, pp={pp}, skip={skip}\n")

print(f"완성: skip = {skip}")
print(f"      skip[pp] 를 읽을 때 항상 pp < pt 였는지 확인 → 이미 채워진 칸만 읽는다 ✅")

### 7. 🟡 [예측] `skip` 배열의 크기가 왜 `len(pat) + 1`?

```python
skip = [0] * (len(pat) + 1)
```

- 패턴 길이가 6인데 배열은 7칸이야. **왜 하나 더 필요할까?**
- `skip[pp]` 에서 `pp`가 가질 수 있는 최댓값은 몇이야? (검색 루프의 `while pp != len(pat)` 조건을 봐)
- `skip[0]` 은 코드에서 **읽히기는 할까?** 어느 분기가 그걸 막고 있지?
- `skip[len(pat)]` 은? 표 만들기 마지막에 이 값이 채워지는데, 검색에서 쓰일까?
  💡 힌트: `pp == len(pat)` 이 되면 `while` 조건이 깨져서 루프를 나가버려.
- 그럼 실제로 쓰이는 칸은 `skip[____] ~ skip[____]` 야.

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
skip = build_skip(pat)
print(f"pat = {pat} (길이 {len(pat)}), skip 크기 = {len(skip)}")
print(f"skip = {skip}\n")

# 검색 중 실제로 읽히는 인덱스 추적
read = set()
def kmp_track_reads(txt, pat, skip):
    pt = pp = 0
    while pt != len(txt) and pp != len(pat):
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif pp == 0: pt += 1
        else:
            read.add(pp); pp = skip[pp]
    return pt - pp if pp == len(pat) else -1

random.seed(0)
for _ in range(300):
    t = ''.join(random.choice('ABCD') for _ in range(random.randint(1, 30)))
    kmp_track_reads(t, pat, skip)
kmp_track_reads(TXT, pat, skip)

print(f"검색 중 실제로 읽힌 skip 인덱스: {sorted(read)}")
print(f"  → skip[0] 읽힘? {0 in read}   (elif pp == 0 분기가 막는다)")
print(f"  → skip[{len(pat)}] 읽힘? {len(pat) in read}   (pp가 len(pat)이면 루프를 나간다)")
print(f"\n  하지만 표 '만들 때'는 skip[{len(pat)}]에 값을 '쓴다':")
print(f"    표 만들기 마지막 단계에서 skip[{len(pat)}] = {skip[len(pat)]}")
print(f"    → 쓰기 때문에 칸이 필요하다. 안 그러면 IndexError!")

---
# 🔍 PART 3 — 검색 손으로 추적하기 (8~10번)

> 표가 완성됐으니 이제 진짜 검색이야.

### 8. 🟢 [손] 교재 예제 완전 추적

```
  idx  0 1 2 3 4 5 6 7 8 9 ...
  txt  Z A B C A B X A C C A D E F
  pat  A B C A B D
skip     -  0  0  1  2  0
```

**표를 채워봐.** (분기는 ① 일치 / ② `pp==0` 불일치 / ③ `pp!=0` 불일치 중 하나)

| 단계 | pt | pp | txt[pt] | pat[pp] | 분기 | 다음 pt, pp |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | Z | A | ② | pt=1, pp=0 |
| 2 | 1 | 0 | A | A | ① | pt=2, pp=1 |
| 3 | 2 | 1 | B | B | ① | pt=3, pp=2 |
| 4 | 3 | 2 | ① | ② | ③ | ④ |
| 5 | 4 | 3 | ⑤ | ⑥ | ⑦ | ⑧ |
| 6 | 5 | 4 | ⑨ | ⑩ | ⑪ | ⑫ |
| 7 | 6 | 5 | ⑬ | ⑭ | ⑮ | ⑯ |
| 8 | | | | | | |

🔥 **7단계가 핵심이야.** `txt[6]='X'` 와 `pat[5]='D'` 가 불일치하는데:
- 브루트 포스라면 `pt`를 **어디로** 되돌렸을까?
- KMP는 `pp = skip[5] = 2` 로 **`pp`만** 바꿔. `pt`는 6 그대로.
- 이게 1번에서 본 **"AB를 다시 비교하지 않는다"** 의 실행 결과야.

*(끝까지 채운 뒤 실행)*

In [ ]:
txt, pat = TXT, PAT
skip = build_skip(pat)
pt = pp = 0
step = 0
print(f"txt = {txt}\npat = {pat}\nskip = {skip[1:len(pat)+1]}  (인덱스 1~{len(pat)})\n")
print("단계| pt | pp |txt[pt]|pat[pp]| 분기                        | 다음")
print("-" * 74)
while pt != len(txt) and pp != len(pat):
    step += 1
    a, b = txt[pt], pat[pp]
    p0, q0 = pt, pp
    if a == b:
        br = "① 일치"
        pt += 1; pp += 1
    elif pp == 0:
        br = "② 불일치 & pp==0 → pt만 +1"
        pt += 1
    else:
        br = f"③ 불일치 → pp=skip[{pp}]={skip[pp]}  (pt 그대로!)"
        pp = skip[pp]
    print(f" {step:2d} | {p0:2d} | {q0:2d} |   {a}   |   {b}   | {br:27s} | pt={pt},pp={pp}")

print(f"\n종료: pt={pt}, pp={pp} → {'찾음' if pp == len(pat) else '못 찾음'}, return {pt-pp if pp==len(pat) else -1}")

### 9. 🟡 [손] 성공하는 예제

이번엔 **찾아지는** 경우로 해보자.

```
  txt  A B A B A B C
  pat  A B A B C
```

- 먼저 `pat = 'ABABC'` 의 **skip 표를 손으로** 만들어봐. (5번 방식)
```
       A   B   A   B   C
skip   -   __  __  __  __
```
- 그다음 검색을 추적해봐. **`pp = skip[pp]` 가 몇 번 일어나?**
- 정답 인덱스는? 26일차 불변식 `pt - pp` 로 계산해봐.
- 🔥 브루트 포스로 같은 입력을 돌리면 비교 횟수가 몇 번이야? KMP는?

*(답을 적은 뒤 실행)*

In [ ]:
def kmp_trace(txt, pat, verbose=True):
    skip = build_skip(pat)
    pt = pp = 0; c = 0
    if verbose:
        print(f"txt = {txt}, pat = {pat}")
        print(f"skip[1..{len(pat)}] = {skip[1:len(pat)+1]}\n")
    while pt != len(txt) and pp != len(pat):
        c += 1
        p0, q0 = pt, pp
        if txt[pt] == pat[pp]:
            br = "일치"; pt += 1; pp += 1
        elif pp == 0:
            br = "pp==0 → pt만 +1"; pt += 1
        else:
            br = f"pp = skip[{pp}] = {skip[pp]} 🔥"; pp = skip[pp]
        if verbose:
            print(f"  {c:2d}: pt={p0}, pp={q0} | '{txt[p0]}' vs '{pat[q0]}' → {br}")
    r = pt - pp if pp == len(pat) else -1
    if verbose:
        print(f"\n  종료: pt={pt}, pp={pp} → return {pt} - {pp} = {r}")
        if r != -1: print(f"  확인: txt[{r}:{r+len(pat)}] = '{txt[r:r+len(pat)]}'")
    return r, c

r, c_kmp = kmp_trace("ABABABC", "ABABC")

def bf_cnt(txt, pat):
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return (pt - pp if pp == len(pat) else -1), c

r2, c_bf = bf_cnt("ABABABC", "ABABC")
print(f"\n비교 횟수: KMP {c_kmp}회 vs 브루트포스 {c_bf}회")

### 10. 🔴 [실험] 🔥 `pt`는 절대 되돌아가지 않는다

교재 314p의 핵심 문장:
> **"KMP법에서 텍스트를 스캔하는 커서 pt는 앞으로 나아갈 뿐 뒤로 되돌아오지 않습니다. 이것은 브루트 포스법에는 없는 특징입니다."**

- 26일차 3번에서 브루트 포스의 `pt`가 **2 → 1로 되돌아가는** 걸 봤지.
- KMP에서는 `pt`가 줄어드는 코드 줄이 **하나도 없어.** 세 분기를 각각 확인해봐:
  - ① 일치 → `pt += 1` (증가)
  - ② `pp == 0` 불일치 → `pt += 1` (증가)
  - ③ `pp != 0` 불일치 → `pt` ____ (변화 없음)
- 그럼 `while` 루프가 무한히 돌 수는 없을까? **③번 분기만 계속 반복되면?**
  💡 힌트: `skip[pp] < pp` 가 항상 성립하면 `pp`는 매번 **줄어들어.** 그리고 `pp >= 0` 이니 언젠가 0이 되고, 그럼 ②로 빠지지.
- 이게 **O(n + m)** 보장의 근거야 (16번).

*(예측을 적은 뒤 실행)*

In [ ]:
def track_pt(txt, pat, algo):
    skip = build_skip(pat)
    pt = pp = 0; hist = []
    while pt != len(txt) and pp != len(pat):
        hist.append(pt)
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif algo == 'kmp':
            if pp == 0: pt += 1
            else: pp = skip[pp]
        else:
            pt = pt - pp + 1; pp = 0
    return hist

t, p = TXT, PAT
h_bf = track_pt(t, p, 'bf')
h_kmp = track_pt(t, p, 'kmp')
print(f"txt = {t}, pat = {p}\n")
print(f"  BF  pt 이력: {h_bf}")
print(f"  KMP pt 이력: {h_kmp}")
print(f"\n  BF  감소한 적 있나? {any(h_bf[i+1] < h_bf[i] for i in range(len(h_bf)-1))}  ← 되돌아감 🔥")
print(f"  KMP 감소한 적 있나? {any(h_kmp[i+1] < h_kmp[i] for i in range(len(h_kmp)-1))}  ← 절대 안 됨 ✅")

print("\n[skip[pp] < pp 가 항상 성립하는가?]")
random.seed(1)
bad = 0
for _ in range(500):
    pt_ = ''.join(random.choice('AB') for _ in range(random.randint(1, 12)))
    sk = build_skip(pt_)
    for i in range(1, len(pt_)+1):
        if not (sk[i] < i): bad += 1
print(f"  랜덤 패턴 500개 검사 → 위반 {bad}회")
print("  → pp는 ③번 분기를 탈 때마다 반드시 줄어든다 = 무한 루프 불가능 ✅")

---
# 💻 PART 4 — 코드 구현 (11~15번)

> 손으로 다 해봤으니 이제 코드로. **모든 변수와 분기가 왜 거기 있는지** 하나씩 짚는다.

### 11. 🟢 [설명] 변수 하나씩 심문하기

```python
def kmp_match(txt, pat):
    pt = 1                          # A
    pp = 0                          # B
    skip = [0] * (len(pat) + 1)     # C
    skip[pt] = 0                    # D
    while pt != len(pat):           # E   ← len(txt)가 아니다!
        ...
    pt = pp = 0                     # F   ← 재사용!
    while pt != len(txt) and pp != len(pat):
        ...
```

**표를 채워봐.**

| 줄 | 무엇인가 | 왜 이 값/이 위치인가 |
|---|---|---|
| A `pt = 1` | ① | ② 왜 0이 아니라 1? |
| B `pp = 0` | ③ | ④ |
| C `skip` 크기 | ⑤ | ⑥ (7번 참고) |
| D `skip[1] = 0` | ⑦ | ⑧ 왜 이 값이 항상 0? |
| E `while pt != len(pat)` | ⑨ | ⑩ 왜 `len(txt)`가 아닐까? |
| F `pt = pp = 0` | ⑪ | ⑫ 같은 변수를 두 번 쓰는 이유 |

- 특히 **F**를 주목해. `pt`, `pp` 가 **표 만들기**와 **검색**에서 **완전히 다른 의미**로 쓰여.
  - 표 만들기: `pt`는 ⑬____ 안의 위치, `pp`는 ⑭____ 안의 위치
  - 검색: `pt`는 ⑮____ 안의 위치, `pp`는 ⑯____ 안의 위치
- 💡 변수를 재사용하지 않고 따로 이름을 붙인다면 어떻게 쓸까? (예: `i`, `j` / `ti`, `pi`)

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
print("[표 만들기 단계에서 pt, pp의 의미]")
print("  pt = 패턴 안의 '뒤쪽' 위치 (표를 채워나가는 커서)")
print("  pp = 패턴 안의 '앞쪽' 위치 (겹침 길이)")
print(f"  두 커서 모두 pat({pat}) 안을 가리킨다\n")

print("[검색 단계에서 pt, pp의 의미]")
print(f"  pt = 텍스트({TXT}) 안의 위치")
print(f"  pp = 패턴({pat}) 안의 위치")
print("  → 같은 이름인데 가리키는 대상이 다르다!\n")

print("[pt = 1 로 시작하는 이유]")
print("  skip[0]은 쓰이지 않고, skip[1]은 항상 0이다 (7번).")
print("  그래서 pt=1부터 시작해 skip[2] 이후를 채워나간다.")
print(f"  실제로: skip[1] = {build_skip(pat)[1]} (패턴 첫 글자에서 실패 → 겹침 0)")

print("\n[변수 이름을 나눠 쓴 버전]")
def kmp_readable(txt, pat):
    m, n = len(pat), len(txt)
    skip = [0] * (m + 1)
    # --- 표 만들기: 패턴 안에서 패턴 찾기 ---
    back, front = 1, 0            # back: 채울 위치, front: 겹침 길이
    skip[back] = 0
    while back != m:
        if pat[back] == pat[front]:
            back += 1; front += 1; skip[back] = front
        elif front == 0:
            back += 1; skip[back] = 0
        else:
            front = skip[front]
    # --- 검색: 텍스트 안에서 패턴 찾기 ---
    ti, pi = 0, 0
    while ti != n and pi != m:
        if txt[ti] == pat[pi]: ti += 1; pi += 1
        elif pi == 0: ti += 1
        else: pi = skip[pi]
    return ti - pi if pi == m else -1

print(f"  kmp_readable('{TXT}', '{pat}') = {kmp_readable(TXT, pat)}")
print(f"  kmp_match   ('{TXT}', '{pat}') = {kmp_match(TXT, pat)}")

### 12. 🟡 [설명] 세 분기의 정체

**표 만들기와 검색, 둘 다 분기가 3개**야. 나란히 놓고 대응시켜봐.

| | 표 만들기 | 검색 | 공통 의미 |
|---|---|---|---|
| ① | `if pat[pt] == pat[pp]` | `if txt[pt] == pat[pp]` | ① |
| ② | `elif pp == 0` | `elif pp == 0` | ② |
| ③ | `else: pp = skip[pp]` | `else: pp = skip[pp]` | ③ |

- ②번 분기가 **왜 따로 필요할까?** ③번의 `skip[pp]` 를 그냥 쓰면 안 되나?
  💡 `pp == 0` 인데 `pp = skip[0]` 을 하면? `skip[0]` 은 0이니 `pp`가 그대로 0이야. 그럼 어떻게 될까?
- ②번에서 표 만들기는 `skip[pt] = pp` 를 **추가로** 하는데 검색은 안 해. 왜?
- ③번은 **`pt`를 안 건드려.** 이게 10번에서 본 "되돌아가지 않는다"의 근거지.

*(답을 적은 뒤 실행)*

In [ ]:
print("[②번 분기를 없애고 ③번만 두면?]")
def kmp_no_branch2(txt, pat, limit=200):
    skip = build_skip(pat)
    pt = pp = 0; g = 0
    while pt != len(txt) and pp != len(pat):
        g += 1
        if g > limit:
            return f"❗ 무한 루프 (pt={pt}, pp={pp})"
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pp = skip[pp]          # 🐛 pp == 0 처리 없음
    return pt - pp if pp == len(pat) else -1

print(f"  결과: {kmp_no_branch2('ZABCABXACCADEF', 'ABCABD')}")
print("\n[왜?]")
print("  pp == 0 인데 불일치 → pp = skip[0] = 0 → 아무것도 안 변함")
print("  pt도 안 변하고 pp도 안 변하니 같은 비교를 영원히 반복 🔥")
print("  → ②번 분기는 'pp를 더 줄일 수 없을 때 pt를 전진시키는' 탈출구다")

### 13. 🟡 [빈칸] 건너뛰기 표 만들기 구현

4~7번에서 손으로 만든 걸 코드로.

**기대 출력**
```
ABCABD → [0, 0, 0, 0, 1, 2, 0]
AAAA   → [0, 0, 1, 2, 3]
ABAB   → [0, 0, 0, 1, 2]
ABCDE  → [0, 0, 0, 0, 0, 0]
정의와 일치: True
```

In [ ]:
def my_build_skip(pat: str) -> list:
    """건너뛰기 표 만들기"""
    pt = ___                          # ① 채워나갈 위치
    pp = ___                          # ② 겹침 길이
    skip = [0] * ___                  # ③ 배열 크기 (7번!)

    skip[pt] = 0
    while ___:                        # ④ 언제까지? (len(txt) 아님!)
        if ___:                       # ⑤ 패턴끼리 비교
            pt += 1
            pp += 1
            skip[pt] = ___            # ⑥ 겹침 길이 기록
        elif ___:                     # ⑦ 더 줄일 수 없을 때
            pt += 1
            skip[pt] = ___            # ⑧
        else:
            pp = ___                  # ⑨ 표를 자기가 참조 (6번!)
    return skip


for p in ['ABCABD', 'AAAA', 'ABAB', 'ABCDE']:
    print(f"{p:7s}→ {my_build_skip(p)}")

# 2번의 '앞뒤 겹침' 정의와 일치하는지 검증
def by_definition(pat):
    sk = [0] * (len(pat) + 1)
    for k in range(1, len(pat)+1):
        sk[k] = max_overlap(pat[:k])[0]
    return sk

random.seed(0)
ok = True
for _ in range(500):
    p = ''.join(random.choice('AB') for _ in range(random.randint(1, 12)))
    if my_build_skip(p) != by_definition(p): ok = False
print(f"정의와 일치: {ok}")

### 14. 🟡 [빈칸] `kmp_match` 완성

**기대 출력**
```
-1
2
랜덤 1000회 검증: 실패 0회
```

In [ ]:
def my_kmp_match(txt: str, pat: str) -> int:
    """KMP법으로 문자열 검색"""
    skip = my_build_skip(pat)

    pt = pp = 0
    while ___ and ___:                # ①② 26일차와 같은 조건
        if ___:                       # ③ 텍스트 vs 패턴
            pt += 1
            pp += 1
        elif ___:                     # ④ 더 되돌릴 곳이 없을 때
            pt += 1                   #    (pp는 이미 0이니 그대로)
        else:
            pp = ___                  # ⑤ 표를 보고 되돌리기 (pt는 그대로!)

    return ___ if ___ else -1         # ⑥⑦ 26일차와 완전히 동일


print(my_kmp_match('ZABCABXACCADEF', 'ABCABD'))
print(my_kmp_match('ABABCDEFGHA', 'ABC'))

fail = 0
for _ in range(1000):
    t = ''.join(random.choice('ABC') for _ in range(random.randint(0, 20)))
    p = ''.join(random.choice('ABC') for _ in range(random.randint(1, 5)))
    if my_kmp_match(t, p) != naive(t, p): fail += 1
print(f"랜덤 1000회 검증: 실패 {fail}회")

### 15. 🔴 [설명] 교재 표와 코드의 `skip` 인덱스 맞추기 🔥

교재 313p의 완성된 표:
```
       A   B   C   A   B   D
       -   0   0   1   2   0
```

그런데 코드로 뽑아보면:
```
skip = [0, 0, 0, 0, 1, 2, 0]
```

**7칸이야.** 교재 표는 6칸인데. 대응을 정확히 맞춰봐.

| 교재 표 칸 | 문자 | 값 | 코드의 `skip[?]` |
|---|---|---|---|
| 1번째 | A | `-` | ① |
| 2번째 | B | 0 | ② |
| 3번째 | C | 0 | ③ |
| 4번째 | A | 1 | ④ |
| 5번째 | B | 2 | ⑤ |
| 6번째 | D | 0 | ⑥ |

- 교재 표의 **k번째 칸 = `skip[k]`** 인지, **`skip[k-1]`** 인지 확인해봐. 4번째 칸(값 1)이 결정적이야.
- 교재는 첫 칸을 `-` 로 뒀는데 **코드에서 `skip[1]` 의 실제 값은 0**이야. 왜 교재는 `-` 로 표시했을까?
  💡 힌트: `pp == 1` 에서 실패하면 `pp = skip[1] = 0` 이 되어 처음부터 다시 시작. 즉 "표를 볼 필요 없이 처음부터"와 같은 뜻이지.
- 그리고 `skip[0]` 은 배열에 존재하지만 **읽히지 않아** (7번). 교재 표에 아예 안 나오는 이유야.
- 💡 **읽는 법 정리**: `skip[k]` = **"패턴 앞 k글자를 맞춘 상태에서 실패했을 때, 되돌아갈 `pp` 값"**

*(답을 적은 뒤 실행)*

In [ ]:
pat = PAT
sk = build_skip(pat)
print(f"pat = {pat}")
print(f"skip = {sk}  (크기 {len(sk)})\n")
print("교재표칸 | 문자 | 교재값 | skip[k] | 의미")
print("-" * 62)
edu = ['-', '0', '0', '1', '2', '0']
for k in range(1, len(pat)+1):
    meaning = f"{k}글자 맞춘 뒤 실패 → pp={sk[k]}로"
    print(f"  {k}번째  |  {pat[k-1]}   |   {edu[k-1]}    |    {sk[k]}    | {meaning}")

print(f"\n→ 교재 표의 k번째 칸 = skip[k]  ✅")
print(f"→ skip[0]={sk[0]} 은 표에 없다 (읽히지 않으므로)")
print(f"→ skip[{len(pat)}]={sk[len(pat)]} 도 검색에선 안 읽힌다 (쓰기용 칸)")

---
# 📐 PART 5 — 성질과 한계 (16~18번)

### 16. 🟡 [실험] 얼마나 빨라졌나

R-2에서 예측한 걸 확인할 시간이야.

- 브루트 포스: **O(n·m)** (26일차 8번)
- KMP: `pt`가 절대 줄지 않고, `pp`도 ③번 분기마다 줄어드니 → **O(?)**

**예측해봐** (최악 패턴 `AAAA...A` 안에서 `AAAB` 찾기, n=1000, m=10):
- 브루트 포스 비교 횟수: ①____
- KMP 비교 횟수: ②____
- 몇 배 차이? ③____

*(예측을 적은 뒤 실행)*

In [ ]:
def bf_cnt2(txt, pat):
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return (pt - pp if pp == len(pat) else -1), c

def kmp_cnt(txt, pat):
    skip = build_skip(pat)
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif pp == 0: pt += 1
        else: pp = skip[pp]
    return (pt - pp if pp == len(pat) else -1), c

print("[최악 패턴]  txt = 'AAA...A',  pat = 'AAA...AB'")
print("    n |  m | 브루트포스 |   KMP  |  배수  |  n+m")
print("-" * 56)
for n in (200, 500, 1000, 2000):
    m = 10
    t = 'A' * n; p = 'A' * (m-1) + 'B'
    _, c1 = bf_cnt2(t, p)
    _, c2 = kmp_cnt(t, p)
    print(f" {n:4d} | {m:2d} | {c1:10d} | {c2:6d} | {c1/c2:5.1f}배 | {n+m:5d}")

print("\n→ KMP의 비교 횟수는 n+m 에 비례한다 → O(n + m) ✅")
print("→ 브루트 포스는 n*m 에 비례 → O(n·m)")

print("\n[교재 예제]")
_, c1 = bf_cnt2(TXT, PAT)
_, c2 = kmp_cnt(TXT, PAT)
print(f"  txt='{TXT}', pat='{PAT}' → BF {c1}회, KMP {c2}회")

### 17. 🔴 [실험] 그런데 KMP는 실무에서 안 쓴다

교재 314p의 충격적인 마무리:
> **"그러나 이 알고리즘은 복잡할 뿐 07-3절의 보이어·무어법 보다 성능 면에서 같거나 오히려 낮은 수준입니다. 따라서 KMP법은 실제 프로그램에서 별로 사용하지 않습니다."**

- 16번에서 KMP가 O(n+m)으로 이론상 최적인데, 왜 실무에서 안 쓸까?
- **평범한 텍스트**(영어 문장, 로그 등)에서 브루트 포스와 KMP를 비교해봐. 차이가 클까?
  💡 힌트: 26일차 8번에서 "실무 텍스트는 첫 글자에서 대부분 걸러진다"고 했지. 그럼 `pp`가 커질 일이 거의 없고, `skip` 표를 쓸 일도 거의 없어.
- 그리고 KMP는 **표를 만드는 비용**(O(m))과 **표를 저장할 메모리**(O(m))가 추가로 들어.
- 07-3 보이어·무어법은 패턴을 **뒤에서부터** 비교해서 한 번에 여러 칸을 건너뛰어. 최선의 경우 **O(n/m)** 까지 가능해.

*(답을 적은 뒤 실행)*

In [ ]:
import string

def bf_cnt2(txt, pat):
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        else: pt = pt - pp + 1; pp = 0
    return (pt - pp if pp == len(pat) else -1), c

def kmp_cnt(txt, pat):
    skip = build_skip(pat)
    pt = pp = 0; c = 0
    while pt != len(txt) and pp != len(pat):
        c += 1
        if txt[pt] == pat[pp]: pt += 1; pp += 1
        elif pp == 0: pt += 1
        else: pp = skip[pp]
    return (pt - pp if pp == len(pat) else -1), c

random.seed(7)

print("[상황 1] 평범한 랜덤 텍스트 (알파벳 26종)")
txt_r = ''.join(random.choice(string.ascii_uppercase) for _ in range(200000))
pat_r = 'QZXJ'
t0 = time.perf_counter(); r1 = bf_match(txt_r, pat_r); e1 = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = kmp_match(txt_r, pat_r); e2 = time.perf_counter() - t0
_, c1 = bf_cnt2(txt_r, pat_r); _, c2 = kmp_cnt(txt_r, pat_r)
print(f"  BF : {e1:.4f}s, 비교 {c1:,}회")
print(f"  KMP: {e2:.4f}s, 비교 {c2:,}회")
print(f"  → 비교 횟수 차이 {abs(c1-c2)}회밖에 안 난다!")

print("\n[상황 2] 알파벳이 2종뿐 (DNA·이진 데이터 같은 경우)")
txt_2 = ''.join(random.choice('AB') for _ in range(200000))
pat_2 = 'AAAAAAAAB'
_, c1 = bf_cnt2(txt_2, pat_2); _, c2 = kmp_cnt(txt_2, pat_2)
print(f"  BF : 비교 {c1:,}회")
print(f"  KMP: 비교 {c2:,}회  ({c1/c2:.2f}배 적음)")

print("\n[상황 3] 최악 (반복 문자열)")
txt_3 = 'A' * 200000
pat_3 = 'A' * 100 + 'B'
_, c1 = bf_cnt2(txt_3, pat_3); _, c2 = kmp_cnt(txt_3, pat_3)
print(f"  BF : 비교 {c1:,}회")
print(f"  KMP: 비교 {c2:,}회  ({c1/c2:.0f}배 적음) 🔥")

print("\n[결론]")
print("  · 평범한 텍스트: 차이 거의 없음 → 표 만드는 비용만 손해")
print("  · 알파벳이 작고 반복 많은 데이터: KMP가 압도적")
print("  · 그래서 KMP는 '최악을 보장해야 할 때' 쓰는 알고리즘")
print("  · 실무 기본값은 보이어·무어 (07-3) 또는 그 변형")

### 18. 🟢 [정리] 문자열 검색 3종 비교

07장을 절반 넘게 왔어. 표를 채워봐.

| 알고리즘 | 비교 방향 | 불일치 시 | 전처리 | 시간 복잡도 | 배운 날 |
|---|---|---|---|---|---|
| 브루트 포스 | 앞→뒤 | `pt` 되돌리고 `pp=0` | 없음 | O(n·m) | 26일 |
| **KMP** | ① | ② | ③ | ④ | **오늘** |
| 보이어·무어 | ⑤ 뒤→앞 | 표 보고 여러 칸 점프 | ⑥ | 최선 O(n/m) | 내일 |

**최종 질문 3개**

1. KMP의 `pt`가 되돌아가지 않는다는 성질이 **왜 O(n+m)을 보장**하지? 두 커서의 움직임으로 설명해봐.
2. 26일차 10번에서 `pt = pt + 1` 버그는 "근거 없이 여러 칸을 밀어" 정답을 놓쳤어. KMP도 여러 칸을 미는데 왜 안 놓칠까? **정확히 무엇이 근거**야?
3. 파이썬 `str.find()`가 KMP를 안 쓰고 **Two-way 알고리즘**을 쓰는 이유는? (26일차 17번 실측 참고)

*(답을 적은 뒤 실행)*

In [ ]:
print("[1] pt와 pp의 움직임으로 보는 O(n+m)")
print("  · pt는 오직 증가만 한다 (①②번 분기) → 최대 n번")
print("  · pp는 ①에서 +1, ③에서 감소 → 증가 총량이 n을 못 넘으니 감소도 최대 n번")
print("  · 표 만들기도 같은 논리로 O(m)")
print("  → 전체 O(n + m)\n")

print("[2] KMP가 미는 '근거'")
p = 'ABCABD'
sk = build_skip(p)
print(f"  pat = {p}, skip = {sk[1:len(p)+1]}")
print(f"  pp=5(=ABCAB 맞춘 상태)에서 실패 → pp = skip[5] = {sk[5]}")
print(f"  근거: 'ABCAB'의 앞 {sk[5]}글자 '{p[:sk[5]]}' == 뒤 {sk[5]}글자 '{p[5-sk[5]:5]}'")
print(f"  → 그 사이 위치에서 시작하는 매칭은 '존재할 수 없음'이 증명된다")
print(f"  → 26일차 버그는 이 증명 없이 그냥 건너뛴 것\n")

print("[3] 왜 str.find는 KMP를 안 쓰나")
print("  · 평범한 텍스트에서 KMP는 브루트 포스 대비 이득이 거의 없다 (17번 상황1)")
print("  · 표 생성 비용 O(m)과 메모리 O(m)이 항상 든다")
print("  · Two-way는 KMP의 최악 보장 + 보이어무어의 점프를 결합한 하이브리드")

print("\n[최종 검증: 세 알고리즘 결과 일치]")
random.seed(11)
fail = 0
for _ in range(2000):
    t = ''.join(random.choice('ABC') for _ in range(random.randint(0, 25)))
    p = ''.join(random.choice('ABC') for _ in range(random.randint(1, 6)))
    a, b, c = bf_match(t, p), kmp_match(t, p), naive(t, p)
    if not (a == b == c): fail += 1
print(f"  랜덤 2000회: 불일치 {fail}회 ✅")

---
---

# ✅ 정답 & 해설

> ⚠️ **표를 손으로 만든 뒤에 내려와.** 특히 4·8번은 직접 그려야 남아.

---

## 🔁 Remind

### R-1
- ① **현재 시도 중인 대조의 시작 위치**
- ② **시작 위치를 오른쪽으로 정확히 1칸 민다** (`pt`를 `pp`만큼 되돌리고 +1, `pp`는 0으로 리셋)
- 26일차 10번 실측: 랜덤 3000회 중 **268회(8.9%)** 오답. 에러 없이 조용히 `-1`을 반환했지.

### R-2
- `pt`가 **불일치해도 줄어들지 않아.** ③번 분기에서는 아예 손도 안 대.
- `pt`는 최대 n번 증가하고, `pp`는 증가한 총량만큼만 감소할 수 있어. 그래서 루프 총 횟수가 **n에 비례**해.
- O(n·m) → **O(n + m)**. 16번에서 실측으로 확인해.

---

## 🎯 PART 1 해설

### 1. 교재 310p 시나리오
- ① **5글자** (`ABCAB`)
- ② `txt[6]` = **X**  ③ `pat[5]` = **D**
- ④ 앞 2글자 = **AB**, ⑤ 뒤 2글자 = **AB** → 같다!
- 그래서 패턴을 **3칸** 밀면 (`5 - 2 = 3`) 패턴의 `AB`가 텍스트의 `AB` 자리에 정확히 겹쳐.
- 💡 이미 `AB`가 맞다는 걸 **아니까** 다시 비교할 필요가 없어. 브루트 포스는 1칸씩 밀면서 이 사실을 매번 다시 확인했지.

> 🔑 KMP의 아이디어는 **"패턴 자신의 반복 구조"** 를 이용하는 거야. 텍스트를 보는 게 아니라 **패턴만 보고** 미리 계산할 수 있어.

---

### 2. 겹치는 문자열

| 문자열 | 최대 겹침 | 그 문자열 |
|---|---|---|
| `A` | ① **0** | |
| `AB` | ② **0** | |
| `ABC` | ③ **0** | |
| `ABCA` | ④ **1** | `A` |
| `ABCAB` | ⑤ **2** | `AB` |
| `AAAA` | ⑥ **3** | `AAA` |
| `ABAB` | ⑦ **2** | `AB` |
| `AABAA` | ⑧ **2** | `AA` |

- **전체 길이는 제외**해야 해. 안 그러면 항상 자기 자신이 답이라 아무것도 못 밀어.
- 정식 용어로는 **경계(border)** 또는 **최장 접두사-접미사(LPS, Longest Prefix Suffix)** 라고 해.

---

### 3. 그림 7-4 읽기

| 실패 위치 | 맞춘 부분 | 겹침 | 다시 시작 |
|---|---|---|---|
| ⓐ 1번째 | (없음) | 0 | 1번째 |
| ⓑ 2번째 | `A` | 0 | 1번째 |
| ⓒ 3번째 | `AB` | 0 | 1번째 |
| ⓓ 4번째 | `ABC` | 0 | ① **1번째** |
| ⓔ 5번째 | `ABCA` | 1 | ② **2번째** |
| ⓕ 6번째 | `ABCAB` | 2 | ③ **3번째** |

- ⓑ~ⓓ가 전부 "1번째부터"인 이유: 2번 표에서 `A`, `AB`, `ABC` 의 겹침이 모두 **0**이야. 재활용할 게 없으니 처음부터 다시.
- **핵심 대응**: `다시 시작할 위치(1-based) = 겹침 + 1`, `0-based 인덱스 = 겹침`
- 그래서 `pp = skip[pp]` 에서 `skip[pp]` 는 곧 **겹침 길이**야.

---

## 📋 PART 2 해설

### 4. 교재 방식 — 패턴끼리 겹치기

- **1칸**: 위 `B` vs 아래 `A` → ① **불일치** → 2번째 문자 값 **0**
- **2칸**: 위 `C` vs 아래 `A` → ② **불일치** → ③ **0**
- **3칸** 🔥: 위 `AB` vs 아래 `AB` → ④ **일치!** → ⑤ **1** (`A`까지), ⑥ **2** (`AB`까지)
- **4칸**: 위 `B` vs 아래 `A` → 불일치 → ⑦ **0**

**완성**
```
       A   B   C   A   B   D
skip   -   0   0   1   2   0
```

**두 방법이 같은 답을 준다**는 게 포인트야:
- 교재 방식: 패턴 두 개를 겹쳐가며 일치하는 지점 찾기
- 2번 방식: 각 접두사의 앞뒤 겹침 계산
- 전자는 **"몇 칸 밀 수 있나"**, 후자는 **"몇 글자 재활용하나"** — 동전의 양면이야.

---

### 5. 다른 패턴들

| 패턴 | skip[1..m] | 특징 |
|---|---|---|
| `AAAA` | `[0, 1, 2, 3]` | 계단식 증가 |
| `ABAB` | `[0, 0, 1, 2]` | |
| `ABCDE` | `[0, 0, 0, 0, 0]` | 전부 0 |
| `AABAAC` | `[0, 1, 0, 1, 2, 0]` | 톱니 모양 |

- **`ABCDE`처럼 전부 0이면** `pp = skip[pp] = 0` 이 되어 항상 처음부터 다시 시작해. **브루트 포스와 거의 같아져.** 단, `pt`는 여전히 안 되돌아가서 그만큼은 이득이야.
- **`AAAA`처럼 계단식이면** `AAAA...` 같은 반복 텍스트에서 압도적으로 유리해. 실패해도 `pp`가 1만 줄어 대부분을 재활용하거든 (17번 상황 3).

> 🔑 **skip 표는 패턴의 "자기 유사성"을 수치화한 것.** 반복이 많은 패턴일수록 값이 크고 KMP가 유리해.

---

### 6. 표 만들기가 KMP를 닮은 이유

**구조 비교**

| | 표 만들기 | 검색 |
|---|---|---|
| 비교 대상 | `pat[pt]` vs `pat[pp]` | `txt[pt]` vs `pat[pp]` |
| 분기 수 | 3개 | 3개 (동일) |
| ③번 동작 | `pp = skip[pp]` | `pp = skip[pp]` (동일) |
| 추가 동작 | `skip[pt] = pp` | 없음 |

- **"패턴 안에서 패턴을 검색한다"** 로 볼 수 있는 이유: 패턴을 1칸 민 자기 자신과 대조하는 거니까, 텍스트 자리에 패턴이 들어간 셈이야. 그래서 코드가 거의 똑같지.
- 🔥 **아직 만드는 중인 표를 참조해도 되는 이유**: `pp < pt` 가 항상 성립하고, `skip[1] ~ skip[pt]` 는 **이미 채워진 상태**야. 즉 **과거의 자기 자신만 참조**해. (동적 계획법의 전형적 패턴이야)
- **`pt = 1`, `pp = 0` 인 이유**: `skip[1]` 은 항상 0으로 확정(첫 글자에서 실패 → 겹침 0)이라 미리 채워두고 `skip[2]` 부터 계산해. `pp = 0` 은 "아직 겹친 게 없다"는 뜻.

---

### 7. `skip` 크기가 `len(pat) + 1`

- `pp` 의 최댓값은 **`len(pat)`** 이야. `while pp != len(pat)` 조건이 그 순간 깨지지만, 표를 **만들 때는** `skip[len(pat)] = pp` 를 **쓴다.**
- **실측**: 검색 중 실제로 읽히는 인덱스는 **1 ~ len(pat)-1** 뿐
  - `skip[0]` → 안 읽힘. `elif pp == 0` 분기가 먼저 잡아채니까.
  - `skip[len(pat)]` → 안 읽힘. `pp == len(pat)` 이면 루프를 나가니까.
- 그런데도 칸이 필요한 건 **표 만들기 마지막에 쓰기 때문**이야. 크기를 `len(pat)` 으로 하면 `IndexError`.

> 🔑 **"읽히지 않지만 써야 해서 필요한 칸"** — 배열 크기 결정할 때 읽기/쓰기를 모두 따져야 한다는 교훈.

---

## 🔍 PART 3 해설

### 8. 교재 예제 완전 추적

```
txt = ZABCABXACCADEF,  pat = ABCABD,  skip[1..6] = [0, 0, 0, 1, 2, 0]
```

| 단계 | pt | pp | txt[pt] | pat[pp] | 분기 | 다음 |
|---|---|---|---|---|---|---|
| 1 | 0 | 0 | Z | A | ② | pt=1, pp=0 |
| 2 | 1 | 0 | A | A | ① | pt=2, pp=1 |
| 3 | 2 | 1 | B | B | ① | pt=3, pp=2 |
| 4 | 3 | 2 | ① **C** | ② **C** | ③ **①일치** | ④ **pt=4, pp=3** |
| 5 | 4 | 3 | ⑤ **A** | ⑥ **A** | ⑦ **①일치** | ⑧ **pt=5, pp=4** |
| 6 | 5 | 4 | ⑨ **B** | ⑩ **B** | ⑪ **①일치** | ⑫ **pt=6, pp=5** |
| 7 | 6 | 5 | ⑬ **X** | ⑭ **D** | ⑮ **③** | ⑯ **pt=6, pp=2** 🔥 |
| 8 | 6 | 2 | X | C | ③ | pt=6, pp=0 |
| 9 | 6 | 0 | X | A | ② | pt=7, pp=0 |

**7단계가 핵심** — 브루트 포스라면 `pt = 6 - 5 + 1 = 2` 로 **되돌아갔을** 자리야. KMP는 `pt = 6` 그대로 두고 `pp` 만 5 → 2로.

최종 결과는 `-1` (텍스트에 `ABCABD` 없음).

---

### 9. 성공하는 예제

`pat = 'ABABC'` 의 skip:
```
       A   B   A   B   C
skip   -   0   1   2   0
```
(`ABA` → 겹침 1, `ABAB` → 겹침 2)

`txt = 'ABABABC'` 검색 → `pp = skip[pp]` 가 **1번** 일어나고, 답은 **2** (`txt[2:7] == 'ABABC'`).

`pt - pp` 불변식은 26일차와 **완전히 동일**하게 작동해. KMP가 바꾼 건 "불일치 시 얼마나 미느냐"뿐이지, 반환값 계산 방식은 그대로야.

---

### 10. 🔥 `pt`는 되돌아가지 않는다

**실측** (`txt=ZABCABXACCADEF`, `pat=ABCABD`)
```
BF  pt 이력: [0,1,2,3,4,5,6, 2,3,4,5,6, 5,6,7,8, 8,9,10,11, 11,12,13]
KMP pt 이력: [0,1,2,3,4,5,6, 6,6, 7,8, 8,9,10,11, 11,12,13]

BF  감소한 적 있나? True   ← 6 → 2 로 되돌아감 🔥
KMP 감소한 적 있나? False  ← 절대 안 됨 ✅
```

**세 분기 확인**
- ① 일치 → `pt += 1`
- ② `pp==0` 불일치 → `pt += 1`
- ③ `pp!=0` 불일치 → **`pt` 변화 없음**

**무한 루프가 불가능한 이유**: `skip[pp] < pp` 가 **항상** 성립해 (랜덤 패턴 500개 검사 → 위반 0회). ③번을 탈 때마다 `pp`가 반드시 줄어들고, `pp >= 0` 이니 언젠가 0이 되어 ②번으로 빠져나가 `pt`가 전진해.

> 🔑 이 두 사실 — **`pt`는 증가만, `pp`는 ③에서 반드시 감소** — 이 O(n+m)의 증명 전부야.

---

## 💻 PART 4 해설

### 11. 변수 심문

| 줄 | 무엇 | 왜 |
|---|---|---|
| A `pt=1` | ① 표를 채워나갈 위치 | ② `skip[1]`은 항상 0으로 확정이라 미리 채우고 `skip[2]`부터 계산 |
| B `pp=0` | ③ 현재 겹침 길이 | ④ 아직 겹친 게 없음 |
| C `len(pat)+1` | ⑤ 표 크기 | ⑥ `skip[len(pat)]`에 **쓰기** 때문 (7번) |
| D `skip[1]=0` | ⑦ 초기값 | ⑧ 첫 글자에서 실패하면 재활용할 게 없음 |
| E `while pt != len(pat)` | ⑨ 표 만들기 종료 | ⑩ 이 단계는 **패턴만** 다루므로 텍스트 길이와 무관 |
| F `pt = pp = 0` | ⑪ 커서 재사용 | ⑫ 표 만들기가 끝났으니 검색용으로 초기화 |

- ⑬ **패턴**, ⑭ **패턴** / ⑮ **텍스트**, ⑯ **패턴**
- 🔥 **같은 이름인데 의미가 완전히 다르다.** 이게 이 코드를 처음 볼 때 가장 헷갈리는 지점이야. 실행 셀의 `kmp_readable` 처럼 `back`/`front`, `ti`/`pi` 로 나누면 훨씬 읽기 쉬워져. 교재가 변수를 재사용한 건 **지면 절약과 대칭성 강조** 때문으로 보여.

---

### 12. 세 분기의 정체

| | 공통 의미 |
|---|---|
| ① | **일치 → 둘 다 전진** |
| ② | **불일치인데 되돌릴 곳이 없음 → 읽기 위치만 전진** |
| ③ | **불일치 → 표를 보고 `pp`만 되돌림 (읽기 위치는 유지)** |

- **②번이 왜 필요한가**: `pp == 0` 인데 ③번을 타면 `pp = skip[0] = 0` 이라 **아무것도 안 변해.** `pt`도 안 변하니 **같은 비교를 영원히 반복** → 무한 루프. 실행 셀에서 확인했지.
  → ②번은 **"더 줄일 수 없을 때의 탈출구"** 야.
- **표 만들기만 `skip[pt] = pp` 를 하는 이유**: 표 만들기의 목적이 **기록**이니까. 검색은 이미 만들어진 표를 **읽기만** 해.

---

### 13. `my_build_skip`

```python
pt = 1                            # ①
pp = 0                            # ②
skip = [0] * (len(pat) + 1)       # ③
while pt != len(pat):             # ④
    if pat[pt] == pat[pp]:        # ⑤
        skip[pt] = pp             # ⑥ (pt, pp 증가 후)
    elif pp == 0:                 # ⑦
        skip[pt] = pp             # ⑧ (= 0)
    else:
        pp = skip[pp]             # ⑨
```
출력: `ABCABD → [0,0,0,0,1,2,0]`, `AAAA → [0,0,1,2,3]`, 정의와 일치 True

---

### 14. `my_kmp_match`

```python
while pt != len(txt) and pp != len(pat):   # ①②
    if txt[pt] == pat[pp]:                 # ③
        pt += 1; pp += 1
    elif pp == 0:                          # ④
        pt += 1
    else:
        pp = skip[pp]                      # ⑤  ← pt 안 건드림!
return pt - pp if pp == len(pat) else -1   # ⑥⑦
```
출력: `-1`, `2`, 랜덤 1000회 실패 0회

**26일차와 다른 곳은 딱 `else` 블록 하나야.** 나머지는 글자 하나까지 똑같아. 그 한 블록이 O(n·m)을 O(n+m)으로 바꾼 거지.

---

### 15. 🔥 교재 표 ↔ 코드 인덱스

| 교재 칸 | 문자 | 값 | 코드 |
|---|---|---|---|
| 1번째 | A | `-` | ① `skip[1]` (실제 값 0) |
| 2번째 | B | 0 | ② `skip[2]` |
| 3번째 | C | 0 | ③ `skip[3]` |
| 4번째 | A | 1 | ④ `skip[4]` |
| 5번째 | B | 2 | ⑤ `skip[5]` |
| 6번째 | D | 0 | ⑥ `skip[6]` |

**교재 표의 k번째 칸 = `skip[k]`** ✅ (4번째 칸의 값 1이 `skip[4]=1`과 일치하는 게 결정적 증거. `skip[3]=0`이니 `skip[k-1]` 해석은 틀려.)

- **교재가 첫 칸을 `-` 로 둔 이유**: `skip[1] = 0` 이라 "처음부터 다시"인데, 이건 표를 참조할 필요조차 없는 자명한 경우거든. 값이 없는 게 아니라 **"표가 의미 없는 칸"** 이라는 표시야.
- `skip[0]` 은 아예 표에 안 나와. 읽히지 않으니까 (7번).

**읽는 법 한 줄 정리**:
> `skip[k]` = **"패턴 앞 k글자를 맞춘 상태에서 실패했을 때 되돌아갈 `pp` 값"**

---

## 📐 PART 5 해설

### 16. 얼마나 빨라졌나

**실측 (최악 패턴)**
```
   n |  m | 브루트포스 |  KMP  | 배수  |  n+m
 200 | 10 |      1919 |   391 | 4.9배 |   210
 500 | 10 |      4919 |   991 | 5.0배 |   510
1000 | 10 |      9919 |  1991 | 5.0배 |  1010
2000 | 10 |     19919 |  3991 | 5.0배 |  2010
```

- KMP의 비교 횟수가 **`n+m`의 약 2배**로 정확히 비례해 → **O(n + m)** ✅
- 브루트 포스는 `n·m` 에 비례 → O(n·m)
- n이 2배가 되면 BF도 KMP도 2배씩 늘지만, **m을 키우면** 격차가 벌어져 (17번 상황 3에서 m=101이면 50배).

**교재 예제**: BF 23회 / KMP 18회 — 짧은 입력이라 차이가 작아.

---

### 17. 그런데 실무에서 안 쓴다

**실측**

| 상황 | BF 비교 | KMP 비교 | 차이 |
|---|---|---|---|
| 랜덤 텍스트(26종), `QZXJ` | 208,088 | 207,774 | **314회 (0.15%)** |
| 2종 알파벳, `AAAAAAAAB` | 2,558 | 1,933 | 1.32배 |
| 최악 `A`×200000, `A`×100+`B` | 20,190,000 | 399,900 | **50배** 🔥 |

**왜 실무에서 안 쓰나**
- **평범한 텍스트에서는 차이가 0.15%** 밖에 안 나. 26일차 8번에서 말한 대로 첫 글자에서 대부분 걸러져서 `pp`가 커질 일이 없거든. 그럼 `skip` 표를 쓸 일도 없지.
- 그런데 **표 생성 비용 O(m)** 과 **메모리 O(m)** 은 **항상** 들어. 이득 없이 비용만 내는 셈.
- 07-3 보이어·무어는 패턴을 **뒤에서부터** 비교해서, 텍스트에 없는 글자를 만나면 **한 번에 m칸** 점프해. 최선 **O(n/m)** — n보다 **적게** 비교할 수 있다는 뜻이야.

**KMP가 빛나는 곳**: DNA 서열(`AGCT` 4종), 이진 데이터, 반복이 많은 로그 등 **알파벳이 작고 자기 유사성이 큰** 데이터. 그리고 **최악을 반드시 보장해야 하는** 시스템.

---

### 18. 문자열 검색 3종 비교

| 알고리즘 | 비교 방향 | 불일치 시 | 전처리 | 시간 복잡도 |
|---|---|---|---|---|
| 브루트 포스 | 앞→뒤 | `pt` 되돌리고 `pp=0` | 없음 | O(n·m) |
| **KMP** | ① **앞→뒤** | ② **`pt` 유지, `pp`만 표대로 축소** | ③ **skip 표 O(m)** | ④ **O(n+m)** |
| 보이어·무어 | ⑤ **뒤→앞** | 표 보고 여러 칸 점프 | ⑥ **이동 표 O(m+σ)** | 최선 O(n/m) |

**최종 질문 답**

**1. O(n+m) 보장**
- `pt`는 ①②번에서만 변하고 **항상 증가** → 최대 n번
- `pp`는 ①에서 +1 되는데, 그 총 증가량이 n을 못 넘어. 그러니 ③에서의 총 감소량도 n 이하
- 두 커서의 총 이동량이 O(n) → 루프 총 횟수 O(n). 표 만들기도 같은 논리로 O(m)

**2. KMP가 미는 근거**
- `skip[5] = 2` 는 `ABCAB` 의 **앞 2글자 `AB` == 뒤 2글자 `AB`** 라는 **증명된 사실**이야.
- 이 사실이 있으면 **그 사이 시작 위치에서는 매칭이 존재할 수 없음**이 따라와. 그래서 건너뛰어도 안전해.
- 26일차 10번의 버그는 이런 증명 없이 그냥 `pp+1` 칸을 건너뛴 거였지. **같은 "여러 칸 밀기"인데 근거의 유무가 알고리즘과 버그를 가른다.**

**3. `str.find`가 KMP를 안 쓰는 이유**
- 평범한 입력에서 이득이 0.15% (17번 상황1)인데 전처리 비용은 항상 발생
- CPython은 **Two-way 알고리즘**을 써 — KMP의 최악 보장과 보이어·무어의 점프를 결합한 하이브리드야
- 26일차 17번에서 `str.find`가 `bf_match`보다 1,000배 빨랐던 게 이 조합 + C 구현의 결과

---

## 📌 핵심 3줄 요약

1. **KMP는 "이미 맞춘 부분을 다시 비교하지 않는다".** 맞춘 부분의 **앞부분과 뒷부분이 겹치는 최대 길이**를 알면, 그만큼은 검증 없이 재활용할 수 있어. 그 값을 미리 계산해둔 게 **건너뛰기 표**야.
2. **`pt`는 절대 되돌아가지 않는다.** 불일치해도 `pp`만 `skip[pp]`로 줄여. `skip[pp] < pp` 가 보장되니 `pp`는 반드시 감소하고, 두 커서의 총 이동량이 O(n)이라 **O(n+m)** 이 나와.
3. **표 만들기 코드와 검색 코드는 구조가 똑같다.** "패턴 안에서 패턴을 검색"하는 것이기 때문이야. 다만 변수 `pt`/`pp`가 두 단계에서 **완전히 다른 의미**로 재사용되니 읽을 때 주의해야 해.

## 🗂️ 스터디 진행 가이드

- 🟢 **(R-1, 1, 3, 4, 11, 18번)**: 전원 필수
  - **4번 표 만들기**가 오늘의 기본기. 패턴 두 개 겹쳐 그리는 걸 직접 해볼 것
  - **11번 변수 심문**은 코드를 처음 볼 때 헷갈리는 걸 전부 정리해줘
- 🟡 **(R-2, 2, 5, 7, 8, 9, 12, 13, 14, 16, 17번)**: 팀 목표선
  - **5번**은 네 패턴을 손으로 다 만들어볼 것. `ABCDE`(전부 0)와 `AAAA`(계단식)의 대비가 핵심
  - **8번 추적표**의 7단계에서 "브루트 포스라면 어디로 갔을까"를 반드시 비교
  - **17번 실측**은 꼭 돌려볼 것. 랜덤 텍스트 0.15% vs 최악 50배
- 🔴 **(6, 10, 15번)**: 도전
  - **6번이 오늘 최고 난도** 🔥 — "만들고 있는 표를 자기가 참조한다"를 이해하면 동적 계획법의 감이 잡혀
  - **15번**은 교재를 볼 때 반드시 겪는 혼란이라 정리해두면 두고두고 편해
- **금요일 코딩테스트 범위**: 🟢🟡 (R-1 ~ 18번)

## 🔗 오늘 회수된 개념들

- **26일차 7번 `pt - pp` 불변식** → 반환값 계산이 완전히 동일 (9번)
- **26일차 10번 "근거 없이 여러 칸 밀기" 버그** → KMP는 근거를 만들어서 민다 (18번 Q2)
- **26일차 8번 O(n·m)** → 오늘 O(n+m)으로 개선 (16번)
- **26일차 17번 `str.find`가 Two-way** → KMP가 실무에서 안 쓰이는 이유 (17~18번)
- **25일차 도수 정렬의 "표 미리 만들기"** → skip 표도 같은 발상 (전처리로 본 계산 줄이기)
- **13~14일차 재귀/기저 조건** → `pp = skip[pp]` 가 무한 루프 안 되는 이유 (10번)
- **23일차 병합 정렬 불변식 / 24일차 힙 가정** → 오늘은 `skip[pp] < pp` 가 종료를 보장 (10번)

---

> **다음 진도 (28일차)**: 07-3 **보이어·무어법**
>
> 교재 315p: 패턴의 **끝 문자에서 시작하여 앞쪽을 향해** 검사해. 텍스트에 아예 없는 글자를 만나면 **한 번에 패턴 길이만큼** 점프하지.
> 오늘까지는 "이미 본 것을 재활용"했다면, 내일은 **"안 본 것을 건너뛴다."** 발상이 정반대야.
> 교재 314p가 예고한 대로 **KMP보다 빠르고 실무에서 실제로 쓰이는** 알고리즘이야.